<a href="https://colab.research.google.com/github/Dragnel25/timeseriesdoan/blob/main/notebooks_03_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from google.colab import drive

# 1. ĐỌC MA TRẬN ĐẶC TRƯNG TỪ FILE 02 TRÊN DRIVE
drive.mount('/content/drive')
df = pd.read_csv('/content/drive/MyDrive/AEP_features.csv', index_col='Datetime', parse_dates=True)

# 2. CHIA TẬP DATA VÀ CHUẨN HÓA CHO XGBOOST & BASELINE
train_end, val_end = int(len(df)*0.7), int(len(df)*0.85)
train, test = df.iloc[:train_end], df.iloc[val_end:]
features = [c for c in df.columns if c != 'Load']

X_train, y_train = train[features], train['Load']
X_test, y_test = test[features], test['Load']

scaler_xgb = StandardScaler()
X_train_scaled = scaler_xgb.fit_transform(X_train)
X_test_scaled = scaler_xgb.transform(X_test)

# 3. HUẤN LUYỆN 3 MÔ HÌNH VÀ LẤY DỰ BÁO
print("1. Đang lấy kết quả Baseline (Naive 24h)...")
y_pred_baseline = test['lag_24'].values

print("2. Đang huấn luyện XGBoost (Mùa vụ)...")
xgb_model = xgb.XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
xgb_model.fit(X_train_scaled, y_train)
y_pred_xgb = xgb_model.predict(X_test_scaled)

print("3. Đang chuẩn bị dữ liệu và huấn luyện LSTM (Deep Learning)...")
data_lstm = df['Load'].values.reshape(-1, 1)
scaler_lstm = StandardScaler()
data_lstm_scaled = scaler_lstm.fit_transform(data_lstm)

X_l, y_l = [], []
look_back = 24
for i in range(len(data_lstm_scaled)-look_back):
    X_l.append(data_lstm_scaled[i:(i+look_back), 0])
    y_l.append(data_lstm_scaled[i+look_back, 0])
X_l, y_l = np.array(X_l), np.array(y_l)
X_l = np.reshape(X_l, (X_l.shape[0], X_l.shape[1], 1))

# Chia tập LSTM khớp với XGBoost
train_size_l = int(len(X_l) * 0.7)
val_size_l = int(len(X_l) * 0.85)
X_train_l, y_train_l = X_l[:train_size_l], y_l[:train_size_l]
X_test_l, y_test_l = X_l[val_size_l:], y_l[val_size_l:]

lstm_model = Sequential([LSTM(50, input_shape=(look_back, 1)), Dense(1)])
lstm_model.compile(loss='mse', optimizer='adam')
lstm_model.fit(X_train_l, y_train_l, epochs=5, batch_size=64, verbose=1)

y_pred_lstm_scaled = lstm_model.predict(X_test_l, verbose=0)
y_pred_lstm = scaler_lstm.inverse_transform(y_pred_lstm_scaled).flatten()

# 4. ĐÓNG GÓI KẾT QUẢ DỰ BÁO CỦA CẢ 3 MÔ HÌNH VÀO DRIVE
# Cắt lấy min_len để đảm bảo DataFrame không bị lỗi nếu LSTM hụt mất 1 dòng
min_len = min(len(y_test), len(y_pred_lstm))

df_results = pd.DataFrame({
    'y_true': y_test.values[:min_len],
    'y_pred_baseline': y_pred_baseline[:min_len],
    'y_pred_xgb': y_pred_xgb[:min_len],
    'y_pred_lstm': y_pred_lstm[:min_len]
}, index=test.index[:min_len])

df_results.to_csv('/content/drive/MyDrive/AEP_predictions.csv')
print("\n=> Đã huấn luyện xong và lưu dự báo của 3 mô hình vào Drive: AEP_predictions.csv")
print("Giờ cậu có thể tắt file 03 này đi, sang file 04 đánh giá!")

Mounted at /content/drive
1. Đang lấy kết quả Baseline (Naive 24h)...
2. Đang huấn luyện XGBoost (Mùa vụ)...
3. Đang chuẩn bị dữ liệu và huấn luyện LSTM (Deep Learning)...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/5
1327/1327 ━━━━━━━━━━━━━━━━━━━━ 19s 13ms/step - loss: 0.0330
Epoch 2/5
1327/1327 ━━━━━━━━━━━━━━━━━━━━ 17s 13ms/step - loss: 0.0098
Epoch 3/5
1327/1327 ━━━━━━━━━━━━━━━━━━━━ 17s 13ms/step - loss: 0.0075
Epoch 4/5
1327/1327 ━━━━━━━━━━━━━━━━━━━━ 19s 15ms/step - loss: 0.0067
Epoch 5/5
1327/1327 ━━━━━━━━━━━━━━━━━━━━ 17s 13ms/step - loss: 0.0063

=> Đã huấn luyện xong và lưu dự báo của 3 mô hình vào Drive: AEP_predictions.csv
Giờ cậu có thể tắt file 03 này đi, sang file 04 đánh giá!
